# 文体を変換する (です・ます調、だ・である調、敬語)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/24_summarization/style_transfer_ja.ipynb)

NLP クックブック 24-12 ／ 要約と平易化

> GPU ランタイム推奨 (ランタイム → ランタイムのタイプを変更 → T4 GPU)

文章の文体を、です・ます調とだ・である調の間で揃え直したい。くだけた文を丁寧な文に言い換えたい。活用形を使った規則と LLM で変換し、文体が変わったか、意味が保たれているか、余計な書き換えがないかを測って比べる。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
# 例文はこのレシピのために作ったもの。常体と敬体の対 30 組 (両方向の変換の正解として使う) と、くだけた文 12 文と敬語の言い換えの一例
PAIRS = [  # (だ・である調, です・ます調)
    ("明日は雨が降る。", "明日は雨が降ります。"),
    ("昨日は図書館で本を読んだ。", "昨日は図書館で本を読みました。"),
    ("彼はまだ駅に着いていない。", "彼はまだ駅に着いていません。"),
    ("その店は日曜日に開かなかった。", "その店は日曜日に開きませんでした。"),
    ("この問題は難しい。", "この問題は難しいです。"),
    ("昨日の試験は易しかった。", "昨日の試験は易しかったです。"),
    ("この部屋はあまり広くない。", "この部屋はあまり広くありません。"),
    ("父は高校の教師だ。", "父は高校の教師です。"),
    ("会議の資料は先週のものだった。", "会議の資料は先週のものでした。"),
    ("それは私の傘ではない。", "それは私の傘ではありません。"),
    ("当時、この町に駅はなかった。", "当時、この町に駅はありませんでした。"),
    ("少し休憩しよう。", "少し休憩しましょう。"),
    ("午後には晴れるだろう。", "午後には晴れるでしょう。"),
    ("締め切りは来週の金曜日である。", "締め切りは来週の金曜日です。"),
    ("窓を開けてもいいか。", "窓を開けてもいいですか。"),
    ("毎朝、駅まで歩いて通っている。", "毎朝、駅まで歩いて通っています。"),
    ("財布を家に忘れてしまった。", "財布を家に忘れてしまいました。"),
    ("来年は海外で働きたい。", "来年は海外で働きたいです。"),
    ("書類は明日までに提出しなければならない。", "書類は明日までに提出しなければなりません。"),
    ("電車が遅れたので、会議に間に合わなかった。", "電車が遅れたので、会議に間に合いませんでした。"),
    ("値段は高いが、品質はとても良い。", "値段は高いですが、品質はとても良いです。"),
    ("先生は「明日は休みだ」と言った。", "先生は「明日は休みだ」と言いました。"),
    ("駅の近くに新しいパン屋ができた。毎朝、行列ができている。", "駅の近くに新しいパン屋ができました。毎朝、行列ができています。"),
    ("彼が来るかどうかは分からない。", "彼が来るかどうかは分かりません。"),
    ("この薬は食後に飲むことになっている。", "この薬は食後に飲むことになっています。"),
    ("彼女は来月、大阪に引っ越すそうだ。", "彼女は来月、大阪に引っ越すそうです。"),
    ("道が混んでいるのは、事故があったからだ。", "道が混んでいるのは、事故があったからです。"),
    ("彼は昨日、熱があったのだ。", "彼は昨日、熱があったのです。"),
    ("明日、もう一度来る。", "明日、もう一度来ます。"),
    ("その映画はまだ見ていないが、評判は聞いている。", "その映画はまだ見ていませんが、評判は聞いています。"),
]
CASUAL = [  # (くだけた文, 敬語の言い換えの一例。正解は 1 つではない)
    ("ちょっと遅れるかも。", "少し遅れるかもしれません。"),
    ("この資料、もう見た？", "この資料はもうご覧になりましたか。"),
    ("明日の会議、出られる？", "明日の会議に出席できますか。"),
    ("それ、すごくいいじゃん。", "それはとても良いですね。"),
    ("今、手が離せないから、あとで電話する。", "今、手が離せないので、後ほどお電話します。"),
    ("悪いけど、ペン貸してくれない？", "申し訳ありませんが、ペンを貸していただけませんか。"),
    ("駅で待ってるね。", "駅でお待ちしています。"),
    ("その件はもう部長に言っといた。", "その件はもう部長に申し上げておきました。"),
    ("資料、メールで送っちゃった。", "資料をメールでお送りしました。"),
    ("何時ごろ来る？", "何時ごろいらっしゃいますか。"),
    ("雨だから、今日はやめとこう。", "雨なので、今日はやめておきましょう。"),
    ("了解、すぐやる。", "承知しました。すぐに取りかかります。"),
]

import re
from fugashi import Tagger

tagger = Tagger()
def split_sents(text):
    return [s for s in re.split(r"(?<=[。！？])", text) if s.strip()]

CASUAL_LEMMAS = {"てる", "でる", "ちゃう", "じゃう", "とく", "どく", "じゃん"}

def ending_style(sentence):
    """文末の文体の判定器: 文末の述語の部分 (最後の自立語から後ろ) に「です」「ます」があれば敬体"""
    ws = [w for w in tagger(sentence) if w.feature.pos1 != "補助記号" and w.surface not in ("「", "」")]
    while ws and ws[-1].feature.pos2 == "終助詞":
        ws.pop()
    tail = []
    for w in reversed(ws):
        if w.feature.pos2 == "接続助詞":               # 「〜ましたので、行く」の前半は見ない
            break
        tail.append(w)
        if w.feature.pos1 in ("名詞", "代名詞", "形状詞", "副詞") or w.feature.pos2 == "一般":
            break
    if any(w.feature.lemma in ("です", "ます") or w.feature.orthBase in ("ください", "ござる") for w in tail):
        return "敬体"
    return "体言止め" if ws and ws[-1].feature.pos1 in ("名詞", "代名詞") else "常体"

def has_polite(text):
    """「」の外に「です」「ます」が 1 つでもあるか (常体にしたい文章に、文中も含めて敬体が残っていないかを見る)"""
    depth, found = 0, False
    for w in tagger(text):
        depth += (w.surface == "「") - (w.surface == "」")
        found |= depth == 0 and w.feature.lemma in ("です", "ます")
    return found

def is_casual(text):
    """くだけた表現 (縮約形、「じゃん」「けど」、疑問の「？」) が残っているか"""
    ws = list(tagger(text))
    return any(w.feature.lemma in CASUAL_LEMMAS or w.surface in ("じゃ", "けど", "？") for w in ws)

# 判定器そのものの確認: 正解の文を正しく判定できるか
plain_ok = sum(ending_style(s) == "常体" for p, _ in PAIRS for s in split_sents(p))
polite_ok = sum(ending_style(s) == "敬体" for _, q in PAIRS + CASUAL for s in split_sents(q))
print(f"常体の文 → 常体: {plain_ok}/{sum(len(split_sents(p)) for p, _ in PAIRS)}  敬体の文 → 敬体: {polite_ok}/{sum(len(split_sents(q)) for _, q in PAIRS + CASUAL)}")

常体の文 → 常体: 31/31  敬体の文 → 敬体: 44/44


In [3]:
# 1. 規則による変換: 形態素解析の活用型・活用形を見て、述語の末尾の語を作り直す
ROW = {"カ": "かきくけこ", "ガ": "がぎぐげご", "サ": "さしすせそ", "タ": "たちつてと", "ナ": "なにぬねの",
       "バ": "ばびぶべぼ", "マ": "まみむめも", "ラ": "らりるれろ", "ワ": "わいうえお"}
TA = {"カ": "いた", "ガ": "いだ", "サ": "した", "タ": "った", "ラ": "った", "ワ": "った", "ナ": "んだ", "バ": "んだ", "マ": "んだ"}

def conj(w, form):
    """動詞 (と「れる・られる」などの助動詞) の基本形と活用型から、ます形・ない形・意志形・た形の語幹を作る"""
    base, ctype = w.feature.orthBase, w.feature.cType
    stem = base[:-1]
    if ctype.startswith("五段"):
        row = ROW[ctype[3]]                                    # "五段-カ行" の「カ」
        if form == "ta":
            return stem + ("った" if w.feature.lemma == "行く" else TA[ctype[3]])
        if base in ("いらっしゃる", "おっしゃる", "くださる", "なさる", "ござる"):
            row = row.replace("り", "い")                       # いらっしゃいます
        return stem + {"renyou": row[1], "mizen": row[0], "ishi": row[4] + "う"}[form]
    if ctype.startswith(("上一段", "下一段", "助動詞-")):        # 「られる」「させる」も一段と同じ
        return stem + {"renyou": "", "mizen": "", "ishi": "よう", "ta": "た"}[form]
    if ctype == "サ行変格":
        return base[:-2] + {"renyou": "し", "mizen": "し", "ishi": "しよう", "ta": "した"}[form]
    if ctype == "カ行変格":
        k = "来" if base.startswith("来") else {"renyou": "き", "mizen": "こ", "ishi": "こ", "ta": "き"}[form]
        return k + {"renyou": "", "mizen": "", "ishi": "よう", "ta": "た"}[form]
    raise ValueError(f"未対応の活用型: {base} {ctype}")

def pos(w, pos1, lemma=None):
    return w.feature.pos1 == pos1 and (lemma is None or w.feature.lemma == lemma)

def verbish(w):
    return pos(w, "動詞") or pos(w, "助動詞") and w.feature.cType in ("助動詞-レル", "助動詞-ラレル", "助動詞-セル", "助動詞-サセル")   # 「出られる」の「られる」

def to_polite_end(t):
    """述語の末尾の語の列 t を常体 → 敬体に。(置き換える末尾の語数, 新しい文字列) を返す。0 語なら後ろに足す"""
    L, P, f = t[-1], (t[-2] if len(t) > 1 else None), t[-1].feature
    if pos(L, "助動詞", "だ"):
        return {"終止形-一般": (1, "です"), "意志推量形": (1, "でしょう")}.get(f.cForm)
    if pos(L, "助動詞", "た") and P is not None:
        if pos(P, "助動詞", "だ"):
            return 2, "でした"
        if P.feature.orthBase == "ある" and len(t) > 2 and pos(t[-3], "助動詞", "だ"):
            return 3, "でした"                                  # であった
        if pos(P, "助動詞", "ない") and len(t) > 2 and verbish(t[-3]):
            return 3, conj(t[-3], "renyou") + "ませんでした"
        if pos(P, "形容詞") and P.feature.orthBase == "ない" and P.feature.pos2 == "非自立可能":
            return 2, "ありませんでした"                         # ではなかった、広くなかった
        if pos(P, "形容詞") or pos(P, "助動詞", "たい"):
            return 0, "です"                                     # 易しかったです
        if verbish(P):
            return 2, conj(P, "renyou") + "ました"
        return None
    if pos(L, "助動詞", "ない") and P is not None and verbish(P):
        return 2, conj(P, "renyou") + "ません"
    if pos(L, "形容詞") and f.orthBase == "ない" and f.pos2 == "非自立可能" and f.cForm.startswith("終止形"):
        return 1, "ありません"
    if verbish(L):
        if f.orthBase == "ある" and P is not None and pos(P, "助動詞", "だ"):
            return 2, "でしょう" if f.cForm == "意志推量形" else "です"     # であろう、である
        if f.cForm.startswith("終止形"):
            return 1, conj(L, "renyou") + "ます"
        if f.cForm == "意志推量形":
            return 1, conj(L, "renyou") + "ましょう"
        return None
    if (pos(L, "形容詞") or pos(L, "助動詞", "たい") or pos(L, "助動詞", "らしい")) and f.cForm.startswith("終止形"):
        return 0, "です"
    return None

def to_plain_end(t):
    """述語の末尾の語の列 t を敬体 → 常体に"""
    s = [w.feature.lemma for w in t[-4:]]
    L, P = t[-1], (t[-2] if len(t) > 1 else None)
    def verb(k):
        return t[-k - 1] if len(t) > k and verbish(t[-k - 1]) else None
    if s[-4:] == ["ます", "ず", "です", "た"] and (v := verb(4)):          # ませんでした
        return (5, "なかった") if v.feature.orthBase == "ある" else (5, conj(v, "mizen") + "なかった")
    if s[-2:] == ["ます", "ず"] and (v := verb(2)):                       # ません (ありません → ない)
        return (3, "ない") if v.feature.orthBase == "ある" else (3, conj(v, "mizen") + "ない")
    if s[-2:] == ["ます", "た"] and (v := verb(2)):                       # ました
        return 3, conj(v, "ta")
    if pos(L, "助動詞", "ます") and (v := verb(1)):
        if L.feature.cForm.startswith("終止形"):
            return 2, v.feature.orthBase
        if L.feature.cForm == "意志推量形":
            return 2, conj(v, "ishi")
    if s[-2:] == ["です", "た"]:
        return 2, "だった"
    if pos(L, "助動詞", "です"):
        if L.feature.cForm == "意志推量形":
            return 1, "だろう"
        if P is not None and (pos(P, "形容詞") or pos(P, "助動詞") and P.feature.lemma != "だ"):
            return 1, ""                                          # 高いです → 高い
        return 1, "だ"
    return None

def convert(text, target):
    """target="polite" (です・ます調) か "plain" (だ・である調) へ。文末と「が・けど」の前 (常体へは接続助詞の前すべて) を変える。「」の中は変えない"""
    words = list(tagger(text))
    pieces = [w.white_space + w.surface for w in words]
    depth, ends = 0, []
    for i, w in enumerate(words):
        depth += (w.surface == "「") - (w.surface == "」")
        nxt = words[i + 1] if i + 1 < len(words) else None
        if depth or w.feature.pos1 in ("補助記号", "助詞"):
            continue
        if nxt is None or nxt.feature.pos2 in ("句点", "終助詞") or (
                nxt.feature.pos2 == "接続助詞" and (target == "plain" or nxt.surface in ("が", "けど", "けれど"))):
            ends.append(i)
    for i in reversed(ends):                                      # 後ろから置き換えれば、前の位置はずれない
        r = (to_polite_end if target == "polite" else to_plain_end)(words[:i + 1])
        if r:
            k, new = r
            if k == 0:
                pieces[i] += new
            else:
                pieces[i + 1 - k:i + 1] = [words[i + 1 - k].white_space + new]
    return "".join(pieces)

LEX = {"ちょっと": "少し", "すごく": "とても", "あとで": "後ほど", "やっぱり": "やはり"}

def casual_to_polite(text):
    """縮約形を戻し、語を置き換えてから敬体にする。尊敬語・謙譲語や、省かれた助詞は扱わない"""
    out, ws = [], list(tagger(text))
    for i, w in enumerate(ws):
        f, s = w.feature, w.surface
        if f.lemma in ("てる", "でる"):
            s = s[0] + "い" + s[1:]                                  # 待ってる → 待っている
        elif f.lemma in ("ちゃう", "じゃう"):
            s = ("て" if s[0] == "ち" else "で") + "しま" + s[2:]   # 送っちゃった → 送ってしまった
        elif f.lemma in ("とく", "どく"):
            s = ("て" if s[0] == "と" else "で") + "お" + s[1:]     # 言っといた → 言っておいた
        elif f.lemma == "じゃん":
            s = "ね"
        elif s == "じゃ" and f.pos1 == "助動詞":
            s = "では"
        elif s in ("けど", "けれど") and f.pos2 == "接続助詞":
            s = "が"
        elif s == "から" and f.pos2 == "接続助詞":
            s = "ので"
            if out and out[-1].strip() == "だ":
                out[-1] = out[-1].replace("だ", "な")                # 雨だから → 雨なので
        elif s == "悪い" and i + 1 < len(ws) and ws[i + 1].surface in ("けど", "けれど"):
            s = "申し訳ない"                                         # 「悪いけど」に限る (「天気が悪い」は変えない)
        elif s == "も" and i >= 1 and ws[i - 1].surface == "か" and (i + 1 == len(ws) or ws[i + 1].feature.pos1 == "補助記号"):
            s = "もしれない"                                         # 遅れるかも → 遅れるかもしれない
        s = LEX.get(s, s)
        out.append(w.white_space + s)
    text = "".join(out)
    question = text.endswith("？")
    text = convert(text.replace("？", "。"), "polite")
    if question and not text.endswith("か。"):
        text = text[:-1] + "か。"
    return text

rule = {"→敬体": [convert(p, "polite") for p, _ in PAIRS], "→常体": [convert(q, "plain") for _, q in PAIRS],
        "くだけ→敬語": [casual_to_polite(c) for c, _ in CASUAL]}
for k in rule:
    print(k, rule[k][:3])

→敬体 ['明日は雨が降ります。', '昨日は図書館で本を読みました。', '彼はまだ駅に着いていません。']
→常体 ['明日は雨が降る。', '昨日は図書館で本を読んだ。', '彼はまだ駅に着いていない。']
くだけ→敬語 ['少し遅れるかもしれません。', 'この資料、もう見ましたか。', '明日の会議、出られますか。']


In [4]:
# 2. 規則を作るときに見ていない文でも試す: 青空文庫の 2 作品 (どちらも著作権の保護期間が終わっている)
# 新美南吉「ごん狐」(です・ます調で語られる) の敬体の文 → 常体、芥川龍之介「羅生門」(だ・である調) の常体の文 → 敬体
# 「」を含む文は除く (何文もある台詞の途中の文は残る。羅生門の老婆の台詞など)
import io, random, urllib.request, zipfile

def aozora(url):
    with urllib.request.urlopen(url) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
    text = z.read(z.namelist()[0]).decode("cp932")
    text = re.split(r"-{50,}", text)[2].split("底本：")[0]
    return re.sub(r"《.+?》|［＃.+?］|｜", "", text).replace("\r\n", "\n").strip()

def pick(url, style, n=50):
    sents = [s.strip() for s in re.split(r"(?<=[。？！])|\n", aozora(url)) if s.strip()]
    sents = [s for s in sents if "「" not in s and "」" not in s and 10 <= len(s) <= 60 and s.endswith("。")
             and ending_style(s) == style]                       # もとから目標の文体の文は、変換しなくても成功になるので除く
    return random.Random(0).sample(sents, n)

gon = pick("https://www.aozora.gr.jp/cards/000121/files/628_ruby_649.zip", "敬体")
rasho = pick("https://www.aozora.gr.jp/cards/000879/files/127_ruby_150.zip", "常体")
print(len(gon), gon[:2], len(rasho), rasho[:2])
rule["ごん狐→常体"] = [convert(s, "plain") for s in gon]
rule["羅生門→敬体"] = [convert(s, "polite") for s in rasho]

50 ['その晩、ごんは、穴の中で考えました。', '兵十はかけよって来ました。'] 50 ['勿論、右の手では、赤く頬に膿を持った大きな面皰を気にしながら、聞いているのである。', 'ただ、おぼろげながら、知れるのは、その中に裸の死骸と、着物を着た死骸とがあるという事である。']


In [5]:
# 3. LLM による変換 (この章の「LLM で文章を要約する (長さと観点を指定する)」と同じモデル)。読み込みを含めて T4 で 3〜5 分ほど
import sentencepiece as spm
import torch
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

set_seed(0)
name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name, padding_side="left")
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる (大規模言語モデルの章の「オープンな LLM とチャットする (チャットテンプレート)」を参照)。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す (特殊トークンやチャットテンプレートはそのまま)
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16 if BF16 else torch.float16, device_map="auto").eval()

PROMPT = {
    "polite": ("次の文を、です・ます調 (敬体) に書き換えてください。文末の表現だけを変え、ほかの言葉や語順は変えないでください。"
               "「」の中はそのままにしてください。書き換えた文だけを出力してください。", "駅前の店は朝九時に開く。", "駅前の店は朝九時に開きます。"),
    "plain": ("次の文を、だ・である調 (常体) に書き換えてください。文末の表現だけを変え、ほかの言葉や語順は変えないでください。"
              "「」の中はそのままにしてください。書き換えた文だけを出力してください。", "この道は夜になると暗くなります。", "この道は夜になると暗くなる。"),
    "keigo": ("次のくだけた文を、職場で目上の人に話すときの丁寧な言い方 (敬語) に書き換えてください。内容は変えないでください。"
              "書き換えた文だけを出力してください。", "その本、もう読んだ？", "その本はもうお読みになりましたか。"),
}

def llm_convert(texts, task, batch_size=16):
    rules, shot_in, shot_out = PROMPT[task]
    outs = []
    for i in range(0, len(texts), batch_size):
        convs = [[{"role": "user", "content": f"{rules}\n\n{shot_in}"}, {"role": "assistant", "content": shot_out},
                  {"role": "user", "content": f"{rules}\n\n{t}"}] for t in texts[i:i + batch_size]]
        enc = tokenizer.apply_chat_template(convs, add_generation_prompt=True, return_tensors="pt", return_dict=True, padding=True).to(llm.device)
        with torch.no_grad():
            out = llm.generate(**enc, max_new_tokens=100, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        outs += [o.strip().split("\n")[0] for o in tokenizer.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)]
    return outs

llm_out = {"→敬体": llm_convert([p for p, _ in PAIRS], "polite"), "→常体": llm_convert([q for _, q in PAIRS], "plain"),
           "くだけ→敬語": llm_convert([c for c, _ in CASUAL], "keigo"),
           "ごん狐→常体": llm_convert(gon, "plain"), "羅生門→敬体": llm_convert(rasho, "polite")}
print(f"GPU メモリの最大使用量: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
for k in llm_out:
    print(k, llm_out[k][:3])

GPU メモリの最大使用量: 7.4 GB
→敬体 ['明日は雨が降ります。', '昨日は図書館で本を読みました。', '彼はまだ駅に着いていません。']
→常体 ['明日は雨が降る。', '昨日は図書館で本を読んだ。', '彼はまだ駅に着いていない。']
くだけ→敬語 ['申し訳ございませんが、少々遅刻するかもしれません。', 'この資料はすでにご覧になりましたか。', '明日の会議にはご出席されますか。']
ごん狐→常体 ['その晩、ごんは、穴の中で考えた。', '兵十はかけよってきた。', 'ごんは物置のそばを離れて、向こうへ行きかけますと、どこかでいわしを売る声がします。']
羅生門→敬体 ['もちろん、右の手では、赤く頬に膿を持った大きな面皰を気にしながら、聞いております。', 'ただ、おぼろげながら、知れるのは、その中に裸の死骸と、着物を着た死骸とがあるということである。', '老婆は、片手に、まだ死骸の頭から奪った長い抜け毛を持ったまま、蟇のつぶやくような声で、口ごもりながら、こんな事を云いました。']


In [6]:
# 4. 評価: 文体の成功率、正解との一致、内容語の保持と追加、正解との編集距離
import pandas as pd

STOP = {"有る", "居る", "為る", "成る", "無い", "下さる", "仕舞う", "置く", "事", "物", "の", "良い"}

def content(text):
    """内容語の語彙素 (表記の揺れ「分かる/わかる」は同じ語彙素になる)。補助的な動詞や形式名詞は除く"""
    return {w.feature.lemma for w in tagger(text) if w.feature.pos1 in ("名詞", "動詞", "形容詞", "副詞", "形状詞") and w.feature.lemma not in STOP}

def edit_distance(a, b):
    d = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev, d[0] = d[0], i
        for j, cb in enumerate(b, 1):
            prev, d[j] = d[j], min(d[j] + 1, d[j - 1] + 1, prev + (ca != cb))
    return d[-1]

TASKS = {  # 課題: (入力, 正解 (なければ None), 目標の文体)
    "→敬体": ([p for p, _ in PAIRS], [[q] for _, q in PAIRS], "敬体"),
    "→常体": ([q for _, q in PAIRS], [[p, p.replace("である", "だ")] for p, _ in PAIRS], "常体"),   # 「である」「だ」どちらも正解
    "くだけ→敬語": ([c for c, _ in CASUAL], None, "敬体"),
    "ごん狐→常体": (gon, None, "常体"),
    "羅生門→敬体": (rasho, None, "敬体"),
}
# 尊敬語・謙譲語の目印 (簡単な正規表現で、すべては拾えない)。「申し訳」と「〜ておく」(ておいた、でおく) の「お」は除く
KEIGO = re.compile(r"ご覧|いらっしゃ|おっしゃ|なさ|いただ|申し(?!訳)|伺|参り|存じ|おり(ま|ませ)|いたし"
                   r"|(?<![てで])お[^\s。、]{1,4}(し|にな|ください)|(?<=[てで])お(?![いかきくけこ])[^\s。、]{1,4}(し|にな|ください)"
                   r"|ご[一-龥]{1,3}(し|にな|さ|ください)")

rows, details = [], []
for method, outputs in [("規則", rule), ("LLM", llm_out)]:
    for task, (srcs, refs, target) in TASKS.items():
        for j, (src, out) in enumerate(zip(srcs, outputs[task])):
            ok = all(ending_style(s) == target for s in split_sents(out))     # 敬体へ: どの文の文末も敬体
            if target == "常体":
                ok = ok and not has_polite(out)                               # 常体へ: 文中の「〜ましたが」なども残っていない
            if task == "くだけ→敬語":
                ok = ok and not is_casual(out)
            cs, co = content(src), content(out)
            details.append({"方法": method, "課題": task, "入力": src, "出力": out, "成功": ok,
                            "一致": float("nan") if refs is None else float(out in refs[j]),
                            "保持": len(cs & co) / max(len(cs), 1), "追加": len(co - cs) / max(len(co), 1),
                            "距離": float("nan") if refs is None else min(edit_distance(out, r) for r in refs[j]),
                            "敬語の語": bool(KEIGO.search(out))})
d = pd.DataFrame(details)
summary = d.groupby(["課題", "方法"], sort=False).agg(件数=("成功", "size"), 文体の成功率=("成功", "mean"), 正解と一致=("一致", "mean"),
                                                   内容語の保持=("保持", "mean"), 内容語の追加=("追加", "mean"), 正解との距離=("距離", "mean"),
                                                   敬語の語を含む=("敬語の語", "mean"))
summary.round(2)

,,件数,文体の成功率,正解と一致,内容語の保持,内容語の追加,正解との距離,敬語の語を含む
課題,方法,,,,,,,
→敬体,規則,30,1.00,1.00,1.00,0.00,0.00,0.00
→常体,規則,30,1.00,1.00,1.00,0.00,0.00,0.00
くだけ→敬語,規則,12,1.00,NaN,0.85,0.16,NaN,0.00
ごん狐→常体,規則,50,1.00,NaN,1.00,0.00,NaN,0.00
羅生門→敬体,規則,50,0.98,NaN,1.00,0.00,NaN,0.00
→敬体,LLM,30,1.00,0.77,1.00,0.02,0.47,0.00
→常体,LLM,30,0.97,0.73,0.98,0.04,2.27,0.00
くだけ→敬語,LLM,12,1.00,NaN,0.39,0.64,NaN,0.83
ごん狐→常体,LLM,50,0.82,NaN,0.98,0.04,NaN,0.00


In [7]:
# 5. 失敗と書き換えすぎの例を読む
pd.set_option("display.max_colwidth", 60)
print("■ 規則: 目標の文体にならなかった文")
print(d[(d.方法 == "規則") & ~d.成功][["課題", "入力", "出力"]].to_string(index=False))
print("■ 規則: 判定器は成功としたが、読むと誤っている文 (老婆の古風な台詞)")
print(d[(d.方法 == "規則") & d.成功 & d.入力.str.contains("であろ。|ぞよ。")][["入力", "出力"]].to_string(index=False))
llm_d = d[(d.方法 == "LLM") & (d.課題 != "くだけ→敬語")]
print(f"\n■ LLM: 目標の文体にならなかった文 ({(~llm_d.成功).sum()} 件のうち先頭 10 件)")
print(llm_d[~llm_d.成功][["課題", "入力", "出力"]].head(10).to_string(index=False))
changed = llm_d[llm_d.成功 & ((llm_d.保持 < 1) | (llm_d.追加 > 0))]
print(f"\n■ LLM: 文体は変わったが、内容語を足した・落とした文 ({len(changed)} 件のうち先頭 6 件)")
print(changed[["課題", "入力", "出力"]].head(6).to_string(index=False))
print("\n■ くだけた文 → 敬語")
for (c, ref), r_out, l_out in zip(CASUAL, rule["くだけ→敬語"], llm_out["くだけ→敬語"]):
    print(f"{c}\n   規則: {r_out}\n   LLM : {l_out}\n   例  : {ref}")

■ 規則: 目標の文体にならなかった文
    課題                           入力                           出力
羅生門→敬体 されば、今また、わしのしていた事も悪い事とは思わぬぞよ。 されば、今また、わしのしていた事も悪い事とは思わぬぞよ。
■ 規則: 判定器は成功としたが、読むと誤っている文 (老婆の古風な台詞)
                            入力                            出力
疫病にかかって死ななんだら、今でも売りに往んでいた事であろ。 疫病にかかって死ななんだら、今でも売りに往んでいた事です。
                   云わぬと、これだぞよ。                  云わぬと、これですぞよ。

■ LLM: 目標の文体にならなかった文 (13 件のうち先頭 10 件)
    課題                                               入力                                                          出力
   →常体                                    来年は海外で働きたいです。 来年は海外で働きたいです。（だ・である調に書き換えると：来年は海外で働きたいね。または、来年は海外で働くつもりです。）
ごん狐→常体       ごんは物置のそばをはなれて、向うへいきかけますと、どこかで、いわしを売る声がします。                   ごんは物置のそばを離れて、向こうへ行きかけますと、どこかでいわしを売る声がします。
ごん狐→常体        つぎの日も、そのつぎの日もごんは、栗をひろっては、兵十の家へもって来てやりました。                     次の日も、その次の日もごんは、栗をひろっては、兵十の家へもって来てやりました。
ごん狐→常体                            それなり、二人はだまって歩いていきました。                                        それなり、二人は黙って歩いていきました。
ごん狐→常体    

- 文体の変換 (style transfer) は、内容を保ったまま書き方だけを変える書き換え。この章の「文章をやさしい日本語に書き換える (テキスト平易化)」と同じく、文体が変わったか、意味が保たれているか、余計な書き換えがないかを分けて測る。です・ます調とだ・である調の変換は文末だけが変わるはずなので、平易化のような含意 (NLI) の判定でなく、内容語の一致と正解との編集距離で書き換えすぎを見つけられる。平易化のプロンプトの「文末は「です」「ます」にする」が守られたかも、上の判定器で確かめられる。
- 規則による方法は、形態素解析の活用型と活用形 (日本語の形態素解析と文分割の章の「動詞・形容詞の活用型と活用形を調べる」) から「行った → 行きました」「広くない → 広くありません」を作る。作例では両方向とも 30/30 が正解と一致したが、規則を作りながら見ていた文なので楽観的な数字である。見ていない青空文庫の文 (変換が要る文だけを選んだ) では、ごん狐 → 常体が 50/50、羅生門 → 敬体が 49/50 で、失敗は古い否定の「ぬ」だった。判定器は成功としたが、老婆の台詞の「であろ → です」(推量が落ちた) や「これだぞよ → これですぞよ」のような誤りもある。
- LLM は「文末の表現だけを変え」と指示しても書き換えすぎる。作例で正解と一致したのは →敬体 23/30、→常体 22/30 で、「から → ため」「分かりません → 不明だ」「歩いて通っています → 歩いている」(「通う」が落ちた) のように、文体は変わったのに内容語が増減した文が 23 あった。
- LLM は変換し損ねも多い。ごん狐 → 常体は 41/50 で、失敗した 9 文のほとんどは文末が「〜ました」「〜ます」のまま、表記の変更 (「だまって → 黙って」) や誤り (「まきついた → まきつくった」) だけが入っていた。「（すでにだ・である調になっていますので、変更は不要です。）」と説明を付け足した文もある。規則なら、文中の「〜ますと」「〜ましたが」もまとめて変えられる。
- くだけた文 → 敬語では、規則は丁寧語と縮約形 (「待ってる」「言っといた」) の復元しかできず、尊敬語・謙譲語は 0/12、省かれた助詞も戻らない。LLM は目印の正規表現で数えて 10/12 で尊敬語・謙譲語を使ったが (「ご出席されますか」のように適否が分かれる表現もある)、内容語の保持は 0.39 で、「遅れる → 遅刻する」「やめとこう → 中止にいたしましょうか」のように語や意図まで変わる。敬語では「見る → ご覧になる」のように語が替わるのが正しいこともあるので、内容語の一致では測れず、含意の判定や人の確認が要る。
- 判定器は正解の文をすべて正しく判定した (常体 31/31、敬体 44/44) が、文末しか見ず、規則と同じ形態素解析器を使うので、規則の誤り (上の「であろ」) を見逃す。くだけた表現の判定 (is_casual) も縮約形や「じゃん」「けど」「？」しか見ず、「かも」や省かれた助詞は拾わないので、規則のくだけ → 敬語の成功率 1.00 は甘い。尊敬語・謙譲語の目印も簡単な正規表現で、すべては拾えない。機械翻訳の章の「タグを付けて訳文の文体を制御する (丁寧さの制御)」も、正規表現の判定器で文体の成功率を測っている。
- 評価は 30〜50 文、1 モデル、貪欲法での結果で、数文の差はばらつきの範囲と考える。実際には、規則で変換して判定器が失敗とした文だけを LLM に回す、LLM の出力を判定器と内容語の一致で検査する、といった組み合わせが考えられる (試していない)。例文はこのレシピのための作例、青空文庫の 2 作品は著者の没年 (1943 年、1927 年) から著作権の保護期間が終わっている。sarashina2.2 は SB Intuitions のモデルで MIT ライセンス。